# LLMOps: Model Evaluation & Comparison Pipeline with MLflow

<a href="https://colab.research.google.com/github/mistralai/cookbook/blob/main/mistral/llmops/llmops_evaluation_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook demonstrates a complete **LLMOps pipeline** for evaluating and comparing Mistral models using [MLflow](https://mlflow.org/). You'll learn how to:

- Track experiments across different models and prompt variants
- Evaluate LLM outputs with custom and built-in metrics
- Compare mistral-small-latest vs mistral-large-latest on standardized tasks
- Version and manage prompts systematically
- Visualize results in an MLflow dashboard

## Installation

In [ ]:
!pip install mistralai==1.5.0 mlflow==2.20.2 pandas==2.2.3 matplotlib==3.9.4

## Setup

In [ ]:
import os
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
from getpass import getpass
from mistralai import Mistral

import mlflow

api_key = getpass("Type your Mistral API Key")
client = Mistral(api_key=api_key)

# Enable MLflow auto-tracing for Mistral
mlflow.mistral.autolog()

## Step 1: MLflow Experiment Setup

We create a dedicated experiment to track all our evaluation runs.

### Considerations:
- MLflow stores all artifacts locally in Colab (or on a remote tracking server in production)
- Each "run" tracks parameters, metrics, and artifacts for reproducibility

In [ ]:
experiment_name = "mistral-model-comparison"
mlflow.set_experiment(experiment_name)

print(f"Experiment '{experiment_name}' created/loaded.")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## Step 2: Evaluation Dataset

We define a standardized evaluation dataset with ground truth answers for multiple task types.

In [ ]:
eval_dataset = pd.DataFrame([
    {
        "task": "summarization",
        "input": "The European Union announced new AI regulations requiring companies to disclose when content is AI-generated. The rules, part of the AI Act, will take effect in 2026 and include penalties up to 7% of global revenue for non-compliance. Tech companies have expressed mixed reactions, with some praising transparency while others warn about implementation challenges.",
        "expected_output": "The EU's AI Act mandates disclosure of AI-generated content, effective 2026, with fines up to 7% of revenue."
    },
    {
        "task": "extraction",
        "input": "Dr. Sarah Chen, a researcher at MIT, published a paper on quantum computing advances. The paper, titled 'Quantum Error Correction at Scale', was published in Nature on March 15, 2025.",
        "expected_output": '{"author": "Dr. Sarah Chen", "affiliation": "MIT", "title": "Quantum Error Correction at Scale", "journal": "Nature", "date": "March 15, 2025"}'
    },
    {
        "task": "reasoning",
        "input": "A train travels from City A to City B at 60 km/h and returns at 90 km/h. What is the average speed for the round trip?",
        "expected_output": "72 km/h"
    },
    {
        "task": "code_generation",
        "input": "Write a Python function that checks if a string is a valid palindrome, ignoring spaces and punctuation.",
        "expected_output": "def is_palindrome(s):\n    cleaned = ''.join(c.lower() for c in s if c.isalnum())\n    return cleaned == cleaned[::-1]"
    },
    {
        "task": "classification",
        "input": "Review: 'The product arrived damaged and customer service was unhelpful. Very disappointed with the experience.'",
        "expected_output": "negative"
    }
])

print(f"Evaluation dataset: {len(eval_dataset)} test cases")
eval_dataset

## Step 3: Prompt Versioning

We define multiple prompt strategies to compare their impact on model performance.

In [ ]:
prompt_versions = {
    "v1_basic": {
        "system": "You are a helpful assistant.",
        "template": "{input}"
    },
    "v2_structured": {
        "system": "You are a precise AI assistant. Always provide concise, structured answers.",
        "template": "Task type: {task}\n\nInput: {input}\n\nProvide a concise response."
    },
    "v3_cot": {
        "system": "You are an expert AI assistant. Think step by step before answering.",
        "template": "Task type: {task}\n\nInput: {input}\n\nThink step by step, then provide your final answer after 'ANSWER:'."
    }
}

## Step 4: Model Evaluation Loop

We run each combination of (model, prompt_version) as a separate MLflow run, tracking all parameters and outputs.

In [ ]:
models_to_compare = ["mistral-small-latest", "mistral-large-latest"]

def run_evaluation(model_name: str, prompt_version: str, prompt_config: dict) -> pd.DataFrame:
    """Run a single evaluation: one model + one prompt version on the full dataset."""
    results = []
    
    with mlflow.start_run(run_name=f"{model_name}_{prompt_version}"):
        # Log parameters
        mlflow.log_param("model", model_name)
        mlflow.log_param("prompt_version", prompt_version)
        mlflow.log_param("system_prompt", prompt_config["system"])
        mlflow.log_param("num_test_cases", len(eval_dataset))
        
        for idx, row in eval_dataset.iterrows():
            user_message = prompt_config["template"].format(
                task=row["task"], input=row["input"]
            )
            
            start_time = time.time()
            response = client.chat.complete(
                model=model_name,
                messages=[
                    {"role": "system", "content": prompt_config["system"]},
                    {"role": "user", "content": user_message}
                ]
            )
            latency = time.time() - start_time
            
            output = response.choices[0].message.content
            tokens_used = response.usage.total_tokens
            
            results.append({
                "task": row["task"],
                "input": row["input"],
                "expected": row["expected_output"],
                "output": output,
                "latency_s": round(latency, 3),
                "tokens": tokens_used,
                "model": model_name,
                "prompt_version": prompt_version
            })
        
        # Log aggregate metrics
        results_df = pd.DataFrame(results)
        mlflow.log_metric("avg_latency_s", results_df["latency_s"].mean())
        mlflow.log_metric("avg_tokens", results_df["tokens"].mean())
        mlflow.log_metric("total_tokens", results_df["tokens"].sum())
        
        # Log results as artifact
        mlflow.log_table(results_df, artifact_file="evaluation_results.json")
    
    return results_df

In [ ]:
all_results = []

for model_name in models_to_compare:
    for prompt_version, prompt_config in prompt_versions.items():
        print(f"\nEvaluating: {model_name} with {prompt_version}...")
        result_df = run_evaluation(model_name, prompt_version, prompt_config)
        all_results.append(result_df)
        print(f"  Avg latency: {result_df['latency_s'].mean():.3f}s | Avg tokens: {result_df['tokens'].mean():.0f}")

combined_results = pd.concat(all_results, ignore_index=True)
print(f"\nTotal evaluations: {len(combined_results)}")

## Step 5: LLM-as-Judge Evaluation

We use Mistral Large to judge the quality of each output compared to the expected output.

### Considerations:
- Using a separate LLM call as judge provides more nuanced evaluation than string matching
- We use a simple 1-5 scoring system for consistency

In [ ]:
def judge_output(task: str, input_text: str, expected: str, actual: str) -> dict:
    """Use Mistral Large as a judge to score output quality."""
    prompt = f"""You are an expert evaluator. Score the following AI output on a scale of 1-5.

Task type: {task}
Input: {input_text}
Expected output: {expected}
Actual output: {actual}

Score criteria:
5 = Perfect match or better than expected
4 = Very good, minor differences
3 = Acceptable, captures main points
2 = Partial, misses key elements
1 = Poor, incorrect or irrelevant

Return ONLY a JSON object with "score" (integer 1-5) and "reasoning" (string)."""

    response = client.chat.complete(
        model="mistral-large-latest",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

# Run LLM-as-Judge on all results
judge_scores = []
for idx, row in combined_results.iterrows():
    score_data = judge_output(row["task"], row["input"], row["expected"], row["output"])
    judge_scores.append(score_data.get("score", 0))
    if idx % 10 == 0:
        print(f"Judging... {idx+1}/{len(combined_results)}")

combined_results["judge_score"] = judge_scores
print(f"\nJudging complete. Average score: {combined_results['judge_score'].mean():.2f}/5")

## Step 6: Log Judge Scores to MLflow

In [ ]:
# Log judge scores as a separate MLflow run for each model+prompt combo
for model_name in models_to_compare:
    for prompt_version in prompt_versions.keys():
        subset = combined_results[
            (combined_results["model"] == model_name) & 
            (combined_results["prompt_version"] == prompt_version)
        ]
        
        with mlflow.start_run(run_name=f"judge_{model_name}_{prompt_version}"):
            mlflow.log_param("model", model_name)
            mlflow.log_param("prompt_version", prompt_version)
            mlflow.log_param("evaluation_type", "llm_as_judge")
            mlflow.log_metric("avg_judge_score", subset["judge_score"].mean())
            mlflow.log_metric("min_judge_score", subset["judge_score"].min())
            mlflow.log_metric("max_judge_score", subset["judge_score"].max())

print("Judge scores logged to MLflow.")

## Step 7: Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Judge scores by model and prompt
score_pivot = combined_results.pivot_table(
    values="judge_score", index="task", columns=["model", "prompt_version"], aggfunc="mean"
)
score_pivot.plot(kind="bar", ax=axes[0], rot=45)
axes[0].set_title("Judge Score by Task, Model & Prompt")
axes[0].set_ylabel("Score (1-5)")
axes[0].legend(fontsize=6)
axes[0].set_ylim(0, 5.5)

# 2. Latency comparison
latency_pivot = combined_results.pivot_table(
    values="latency_s", index="model", columns="prompt_version", aggfunc="mean"
)
latency_pivot.plot(kind="bar", ax=axes[1], rot=0)
axes[1].set_title("Average Latency by Model & Prompt")
axes[1].set_ylabel("Latency (seconds)")

# 3. Cost estimation
cost_map = {"mistral-small-latest": 0.0000001, "mistral-large-latest": 0.000002}
combined_results["estimated_cost"] = combined_results.apply(
    lambda r: r["tokens"] * cost_map.get(r["model"], 0), axis=1
)
cost_pivot = combined_results.pivot_table(
    values="estimated_cost", index="task", columns="model", aggfunc="sum"
)
cost_pivot.plot(kind="bar", ax=axes[2], rot=45)
axes[2].set_title("Estimated Cost by Task & Model")
axes[2].set_ylabel("Cost ($)")

plt.tight_layout()
plt.show()

## Step 8: Summary Report

In [ ]:
summary = combined_results.groupby(["model", "prompt_version"]).agg({
    "judge_score": "mean",
    "latency_s": "mean",
    "tokens": ["mean", "sum"],
    "estimated_cost": "sum"
}).round(4)

print("=== EVALUATION SUMMARY ===\n")
print(summary.to_string())

# Best configuration
best_idx = combined_results.groupby(["model", "prompt_version"])["judge_score"].mean().idxmax()
print(f"\nBest configuration: {best_idx[0]} with {best_idx[1]} (avg score: {combined_results.groupby(['model', 'prompt_version'])['judge_score'].mean().max():.2f}/5)")

## Step 9: MLflow Dashboard

To view the full MLflow dashboard with all runs, metrics, and artifacts:

```bash
mlflow ui --port 5000
```

Then open http://localhost:5000 in your browser.

## Summary

This notebook implemented a full LLMOps evaluation pipeline:

| Step | Tool | Purpose |
|------|------|---------|
| Experiment tracking | `mlflow.set_experiment()` | Organize evaluation runs |
| Auto-tracing | `mlflow.mistral.autolog()` | Automatic trace capture |
| Parameterization | `mlflow.log_param()` | Track model/prompt versions |
| LLM-as-Judge | Mistral Large | Score outputs 1-5 with reasoning |
| Visualization | matplotlib + MLflow UI | Compare results |

### Considerations:
- **Production deployment**: Replace local MLflow with a remote tracking server (e.g., Databricks, self-hosted)
- **CI/CD integration**: Wrap this pipeline in a script for automated evaluation on model updates
- **Custom metrics**: Define domain-specific evaluation metrics beyond the 1-5 judge score
- **Prompt registry**: Use MLflow's Model Registry to version and stage prompts alongside models
- **A/B testing**: Use this framework to compare model versions before production rollout